# The Split That Inflated the Result by 24 Points

**Computational Text Analysis Pack · Module 13 — Case Studies**

*Technique:* Group leakage; why a random split flatters an authorship model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/16-case-studies/13b_the_leak.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Here is the mistake, and it is one of the most common in computational literary
studies.

To train on passages you must first cut novels into passages. The obvious next
step is to shuffle all the passages and split them into train and test. It is
what `train_test_split` does by default, and it is wrong here.

Passages from *Things Fall Apart* share Okonkwo, Umuofia, yams, the egwugwu. If
some land in training and others in test, the model can score beautifully by
recognising **the novel** — and never learn anything about how Achebe writes.

## The measured cost

Both models below are identical: same architecture, same passages, same
hyperparameters. Only the split differs.

| Split | Accuracy |
|---|---|
| Passages shuffled across novels | **0.88** |
| Whole novels held out | **0.64** |
| Chance (7 authors) | 0.14 |

**24 points of inflation.** The first number is not a small overstatement of the
second; it is a different claim about a different task.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# The same trap, reproduced here on a corpus we can ship. `work_id` in the
# colonial corpus plays the part `book` played in the novels.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score

counts = df_colonial["author"].value_counts()
multi = counts[counts >= 2].index
sub = df_colonial[df_colonial["author"].isin(multi)].reset_index(drop=True)
# Authors need 2+ distinct works or a group split is impossible.
ok = sub.groupby("author")["work_id"].nunique()
sub = sub[sub["author"].isin(ok[ok >= 2].index)].reset_index(drop=True)
print(f"{len(sub):,} passages, {sub['author'].nunique()} authors, "
      f"{sub['work_id'].nunique()} works")

In [ ]:
def score(train_idx, test_idx, label):
    vec = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=8000)
    A = vec.fit_transform(sub["text"].iloc[train_idx])
    B = vec.transform(sub["text"].iloc[test_idx])
    clf = LogisticRegression(max_iter=1000).fit(A, sub["author"].iloc[train_idx])
    acc = accuracy_score(sub["author"].iloc[test_idx], clf.predict(B))
    print(f"   {label:<38}{acc:.3f}")
    return acc

print("Same data, same model, two splits:\n")
tr, te = train_test_split(np.arange(len(sub)), test_size=0.3, random_state=0,
                          stratify=sub["author"])
leaky = score(tr, te, "passages shuffled (LEAKY)")

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
tr2, te2 = next(gss.split(sub, groups=sub["work_id"]))
honest = score(tr2, te2, "whole works held out (honest)")

chance = sub["author"].value_counts(normalize=True).max()
print(f"   {'majority-class baseline':<38}{chance:.3f}")
print(f"\n   inflation from the leaky split: {leaky - honest:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ["chance", "whole works\n(honest)", "shuffled passages\n(leaky)"]
vals = [chance, honest, leaky]
cols = ["grey", "seagreen", "indianred"]
ax.bar(bars, vals, color=cols)
for i, v in enumerate(vals):
    ax.text(i, v + .01, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylabel("accuracy")
ax.set_title("The split decides the number")
ax.set_ylim(0, 1.05)
plt.tight_layout()

## How to tell whether it happened to you

The tell is a **validation score far above your test score**, or a test score
that looks too good for the task. Three checks worth running on any corpus with
a natural grouping — book, author, newspaper issue, manuscript, speaker:

1. Is there a group variable? If passages, pages or utterances came from a
   larger unit, that unit is your group.
2. Does any group appear on both sides of the split? One line of code.
3. Does the score drop when you split by group? If it drops a lot, the original
   number was measuring memorisation.

In [ ]:
# Check 2, as a reusable assertion.
def leak_check(frame, train_idx, test_idx, group_col):
    a = set(frame[group_col].iloc[train_idx])
    b = set(frame[group_col].iloc[test_idx])
    shared = a & b
    print(f"groups in train {len(a)}, in test {len(b)}, in BOTH {len(shared)}")
    return shared

print("random split:")
bad = leak_check(sub, tr, te, "work_id")
print("\ngroup split:")
good = leak_check(sub, tr2, te2, "work_id")
print()
print("The first shares works across the split; the second shares none.")
print("Put this assertion in your pipeline and the mistake cannot recur.")

### 🔧 Your turn 1

Run the same comparison grouping by `author` instead of `work_id`. What question is that model answering?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Grouping by author makes the test set contain authors never seen in training, so
the model cannot possibly name them — accuracy collapses toward zero. That is
not a bug: it shows that "attribute this passage to one of these seven authors"
and "identify an unseen author's style" are different tasks, and only the first
is what this pipeline does. Being explicit about which one you are claiming is
most of what separates a careful attribution study from an overclaim.

</details>